In [101]:
import torch
import torch.nn as nn
from torch import Tensor

def block_attn_res(blocks: list[Tensor], partial_block: Tensor, proj: nn.Linear, norm: nn.Module) -> Tensor:
    """
    Rewritten Inter-block attention without einsum.
    """
    # 1. Stack all representations
    # V shape: [N+1, B, T, D] (where N is the number of completed blocks)
    # 1st iter: blocks = [], partial_block = (B,T,D)
    V = torch.stack(blocks + [partial_block]) # V = (B,T,D) oder (1,B,T,D)

    # so let's say we have V number of blocks, and we want to attend to the previous blocks but in a attention way
    
    # so each block
    # what is K? =>

    
    # 2. Normalize
    # K shape: [N+1, B, T, D]
    # norm does just a normal normalization along the dimension D
    # so K probably is the key that is emitted from each block in V
    K = norm(V) # Norm does an RMSNorm on dimension: D current shape: (B,T,D) oder (1,B,T,D)
    
    # 3. Compute attention logits
    # Assuming proj is an nn.Linear(D, 1), proj(K) yields [N+1, B, T, 1].
    # Squeezing the last dimension gives [N+1, B, T].
    # (This matches the original einsum dot product across 'd')
    # what squeeze does: removes the dimension of size 1, so from (N+1, B, T, 1) to (N+1, B, T)
    # Here we do a projection from each block previously
    # logits become (N+1, b, t) => projection is from D to 1
    # h = logits.softmax along the dimension 0, wtf does this do
    # so a softmax is c
    logits = proj(K).squeeze(dim=-1) # we do a proj from D to D, then we do a squeeze 
    
    # 4. Compute attention weights
    # Softmax across the block dimension (dim=0)
    # attn_weights shape: [N+1, B, T]
    attn_weights = logits.softmax(dim=0)
    
    # 5. Apply weights and sum
    # Unsqueeze to [N+1, B, T, 1] so it broadcasts with V [N+1, B, T, D]
    # Multiply element-wise, then sum across the block dimension (dim=0)
    # h shape: [B, T, D]
    h = (attn_weights.unsqueeze(-1) * V).sum(dim=0)
    # h at the end is (B,T,D)
    return h

# The forward pass doesn't use einsum, but here it is for completeness
def forward(self, blocks: list[Tensor], hidden_states: Tensor) -> tuple[list[Tensor], Tensor]:
    # in the first iteration: blocks is [] and hidden_states = (B,T,D) -> embedding of a token
    partial_block = hidden_states # partial_block = (B,T,D)
    
    # 1. Block Attention before Self-Attention
    h = block_attn_res(blocks, partial_block, self.attn_res_proj, self.attn_res_norm)
    
    # 2. Check Block Boundary
    if self.layer_number % (self.block_size // 2) == 0:
        blocks.append(partial_block)
        partial_block = None
        
    # 3. Self-Attention Layer
    attn_out = self.attn(self.attn_norm(h))
    partial_block = partial_block + attn_out if partial_block is not None else attn_out
    
    # 4. Block Attention before MLP
    h = block_attn_res(blocks, partial_block, self.mlp_res_proj, self.mlp_res_norm)
    
    # 5. MLP Layer
    mlp_out = self.mlp(self.mlp_norm(h))
    partial_block = partial_block + mlp_out
    
    return blocks, partial_block

In [ ]:
# so like, when we do normally a residual connection we do:
# (B,T,D) -> original token embedding + previous layer info + (B,T,D) => output from the residual layer
# so we have some info from all the previous layers: x (B,T,D) => each token has learned something
# we had a residual pathway where we did some computation: out = self.MLP(x) 
# now we combine these and represent again in x i.e x = x + out
# now x becomes the input to the next layer
# here x = 1*x + 1*out (but we probably do not want 1, we want some learnable weightage to residual connections)
# what this does fundamentally: 
# keys are the outputs from each prevous layer
# query: trainable D dimension parameter
# intially, keys and values are the previous layer outputs # V is of size (n,b,t,d)
# we combine key and query -> here query is a D dimension vector
# query basically for a token looks at its entire representation from a layer and projects to a single value
# # so now we have (N,B,T) => a value representing token across all the previous layers
# we now take a softmax across all the values of the token across all the layers 
# 
# so when we combine query and key: we get (N,B,T) x (N,B,T,D) value -> we get (B,T,D )
# 4x8 with 4x8x16 -> so each 16 values of a token representation will get scaled with corresponding learned layer softmax value
# for each token along each dimension D, it adds 


In [ ]:
# let's take a simple example, a single sequence of tokens (T,D)
# we have some previous N layers:
# so, let's stack the output from each layer: (N,T,D)
# if we were normal attention residual we would have just concatenated them all along the dimension 0
# instead we want to learn to weight each token's contribution from each layer
# how can we do that?
# we use key, query and value approach
# each token is already emitting a value from each layer => This becomes V
# K = norm(V) -> we want to normalize the value of each token's dimensional value
# Next we do a query multiplication as to sum what the token represents -> query -> linear_proj (D)
# out becomes = query(K) => (N,T,1) -> N layers, T tokens, each token represented by a single number
# Now we do a softmax -> for all the token values across each layer
# Now we do a dot product with value V
# this represents how much each token representation from each layer output contributes to current layer
# So the output after softmax is (N,T)
# Value V is (N,T,D)
# We want to do a element wise multiplication here => out.unsqueeze(-1) > N,T,1
# Now we do element wise multiplication: V*out => token's softmax contribution is applied to each of its dimensional value
# and then we take sum across all the layers for the token
# out.sum(dim=0)

In [ ]:
# The forward pass doesn't use einsum, but here it is for completeness
def forward(self, blocks: list[Tensor], hidden_states: Tensor) -> tuple[list[Tensor], Tensor]:
    # in the first iteration: blocks is [] and hidden_states = (B,T,D) -> embedding of a token
    partial_block = hidden_states # partial_block = (B,T,D)
    
    # 1. Block Attention before Self-Attention layer
    h = block_attn_res(blocks, partial_block, self.attn_res_proj, self.attn_res_norm)
    
    # 2. Check Block Boundary
    if self.layer_number % (self.block_size // 2) == 0:
        blocks.append(partial_block)
        partial_block = None
        
    # 3. Self-Attention Layer
    attn_out = self.attn(self.attn_norm(h))
    partial_block = partial_block + attn_out if partial_block is not None else attn_out
    
    # 4. Block Attention before MLP
    h = block_attn_res(blocks, partial_block, self.mlp_res_proj, self.mlp_res_norm)
    
    # 5. MLP Layer
    mlp_out = self.mlp(self.mlp_norm(h))
    partial_block = partial_block + mlp_out
    

In [2]:
2%4

2

In [102]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # x: (..., D)
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return (x / rms) * self.weight

In [103]:
class TransformerLayer(nn.Module):
    def __init__(self, dim, n_heads, mlp_mult, block_size, layer_number):
        super().__init__()
        self.layer_number = layer_number
        self.block_size   = block_size          # in terms of ATTN+MLP pairs

        # block attention gates
        self.attn_res_proj = nn.Linear(dim, dim, bias=False)
        self.attn_res_norm = RMSNorm(dim)
        self.mlp_res_proj  = nn.Linear(dim, dim, bias=False)
        self.mlp_res_norm  = RMSNorm(dim)

        # standard transformer ops
        self.attn_norm = RMSNorm(dim)
        self.mlp_norm  = RMSNorm(dim)
        self.attn      = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.mlp       = nn.Sequential(
            nn.Linear(dim, dim * mlp_mult),
            nn.GELU(),
            nn.Linear(dim * mlp_mult, dim),
        )

    def forward(self, blocks, partial_block):
        # ── pre-attention gate ──────────────────────────────────
        # partial_block: (B, T, D)
        h = block_attn_res(
            blocks, partial_block,
            self.attn_res_proj, self.attn_res_norm
        )                                            # (B, T, D)

        # ── block boundary ──────────────────────────────────────
        # each layer counts as 1; block_size is number of layers per block
        if self.layer_number % self.block_size == 0:
            blocks = blocks + [partial_block]        # freeze snapshot
            partial_block = torch.zeros_like(partial_block)

        # ── self-attention ──────────────────────────────────────
        normed = self.attn_norm(h)                   # (B, T, D)
        attn_out, _ = self.attn(normed, normed, normed)  # (B, T, D)
        partial_block = partial_block + attn_out     # (B, T, D)

        # ── pre-MLP gate ────────────────────────────────────────
        h = block_attn_res(
            blocks, partial_block,
            self.mlp_res_proj, self.mlp_res_norm
        )                                            # (B, T, D)

        # ── MLP ─────────────────────────────────────────────────
        mlp_out = self.mlp(self.mlp_norm(h))         # (B, T, D)
        partial_block = partial_block + mlp_out      # (B, T, D)

        return blocks, partial_block

In [104]:
class BlockAttnTransformer(nn.Module):
    def __init__(self, vocab_size, dim, n_layers, n_heads, mlp_mult, block_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
        self.layers = nn.ModuleList([
            TransformerLayer(dim, n_heads, mlp_mult, block_size, i)
            for i in range(n_layers)
        ])
        self.out_norm = RMSNorm(dim)
        self.lm_head  = nn.Linear(dim, vocab_size, bias=False)

    def forward(self, x):
        # x: (B, T) — token indices

        partial_block = self.embed(x)   # (B, T, D)
        blocks = []                     # no past blocks at start

        for layer in self.layers:
            blocks, partial_block = layer(blocks, partial_block)
            # blocks: list of (B, T, D), grows at each boundary
            # partial_block: (B, T, D), accumulates residuals within block

        out = self.out_norm(partial_block)  # (B, T, D)
        logits = self.lm_head(out)          # (B, T, vocab_size)
        return logits


In [24]:
blocks = []
partial_blocks = torch.randn([4,8,32])

In [26]:
blocks = blocks + [partial_blocks]
print(len(blocks))

2


In [28]:
new_p = torch.randn([4,8,32])
blocks = blocks + [new_p]
print(len(blocks))

4


In [29]:
new_p = torch.randn([4,8,32])
news = torch.stack(blocks + [new_p])
news.shape

torch.Size([5, 4, 8, 32])

In [9]:
import torch
import torch.nn as nn
x = torch.randn([3,3,3])
rms = nn.RMSNorm(3)
print(x)
print(rms(x))

tensor([[[-0.3265, -0.0649, -0.1141],
         [ 0.4113,  0.6263,  0.7175],
         [-0.8662,  0.3801, -1.4094]],

        [[ 1.1426, -2.2071,  1.2942],
         [-1.7173, -0.5887,  1.4932],
         [-1.1487, -0.9611,  0.3453]],

        [[ 0.4629,  1.2674,  0.5301],
         [-0.5042,  1.5452, -0.6618],
         [ 0.1793, -0.5714,  0.6146]]])
tensor([[[-1.6070, -0.3194, -0.5618],
         [ 0.6867,  1.0457,  1.1979],
         [-0.8839,  0.3879, -1.4382]],

        [[ 0.7063, -1.3643,  0.8000],
         [-1.2654, -0.4338,  1.1003],
         [-1.2945, -1.0830,  0.3891]],

        [[ 0.5531,  1.5142,  0.6334],
         [-0.4976,  1.5251, -0.6531],
         [ 0.3620, -1.1534,  1.2404]]], grad_fn=<MulBackward0>)


In [ ]:
# this is a forward for a single layer of transformer 

def block_attn_res(partial_block, blocks, norm, proj):
    V = torch.stack(partial_block, blocks) # this becomes a N,B,T,D
    # next we need queries and keys
    K = norm(V) # normalize on the values of dimension D for each token
    # Now we want to project the token values from Dimension D to a single dimension
    logits = proj(K).squeeze(dim=-1) # logits become (N,B,T)
    softm = logits.softmax(dim=0) # so across each layer info for each token representation we want to softmax
    # so as to weigh how much each layer's token representation contributes
    out = V * softm.unsqueeze(-1).sum(dim=0)
    return out


class TransformerLayer(nn.Module):
    def __init__(self, super, layer_number: int, dim):
        super().__init__()
        self.layer_number = layer_number
        self.RMSNorm = nn.RMSNorm()
        self.proj = nn.Linear(dim, 1)
        
    def forward(self, blocks, partial_block):
        # we want to pass something into the attention
        # how can that something be calculated:
        # something can be calculated based off block and partial_block
        # partial_block is the current running representation of x
        # but we want to use representations from previous block layers
        x = block_attn_res(partial_block, blocks, self.RMSNorm, self.proj)

        if (self.layer_number%(self.block_size //2 ) == 0):
            blocks.append(partial_block)
            partial_block = None

        out = self.attn(self.norm(x))
        if partial_block is None:
            partial_block = out
        else:
            partial_block += out

        x = block_attn_res(partial_block, blocks, self.RMSNorm,self.proj)

        out = self.MLP(self.norm(x))
        partial_block += out

        return blocks, partial_block

        
        

In [105]:
import torch
a = torch.tensor([[[1,2,3], [4,5,6]], [[4,1,1], [1,2,3]]], dtype=float)

In [106]:
a.shape

torch.Size([2, 2, 3])

In [107]:
b = a.squeeze(dim=-1)
b

tensor([[[1., 2., 3.],
         [4., 5., 6.]],

        [[4., 1., 1.],
         [1., 2., 3.]]], dtype=torch.float64)

In [108]:
import torch.nn as nn 
rms = nn.RMSNorm([3])
print(a)
rms(a)

tensor([[[1., 2., 3.],
         [4., 5., 6.]],

        [[4., 1., 1.],
         [1., 2., 3.]]], dtype=torch.float64)


tensor([[[0.4629, 0.9258, 1.3887],
         [0.7895, 0.9869, 1.1843]],

        [[1.6330, 0.4082, 0.4082],
         [0.4629, 0.9258, 1.3887]]], dtype=torch.float64,
       grad_fn=<MulBackward0>)

In [109]:
print(a)
a.softmax(dim=0)

tensor([[[1., 2., 3.],
         [4., 5., 6.]],

        [[4., 1., 1.],
         [1., 2., 3.]]], dtype=torch.float64)


tensor([[[0.0474, 0.7311, 0.8808],
         [0.9526, 0.9526, 0.9526]],

        [[0.9526, 0.2689, 0.1192],
         [0.0474, 0.0474, 0.0474]]], dtype=torch.float64)

In [66]:
a = torch.randn([4,4])
a

tensor([[-0.8663,  0.3451, -0.8398, -0.2565],
        [-2.5467,  1.1316, -0.2155,  0.6191],
        [-0.1070, -0.2592,  1.2676,  1.1696],
        [-0.3070, -0.6829, -0.5335,  0.1623]])

In [69]:
# where do we do RMS Norm Anyway? -> we do it before query multiplication
# what does query do -> 
b = torch.randn([4,4,2])

In [80]:
x = a.softmax(dim=0)
print(x.shape, b.shape)
out = (x.unsqueeze(-1) * b)
out = out.sum(dim=0)
print(out.shape)

torch.Size([4, 4]) torch.Size([4, 4, 2])
torch.Size([4, 2])


In [124]:
5%(4//2)

1

In [ ]:
# layer % block_size == 0 => start a new block
# so if we have 16 layers, and block size of 4
# we want at max 4 or 5 blocks to be trained
# but each layer in itself has 2 layers within it -> so total number of layers get mult by 2
# we need to keep check
# layers are numbered from 1 .. N
# let's say block_size = 8 or b
# each layer in itself has 2 layers -> attn and MLP so block_size = 8 means -> 4 attn blocks
# self.current_layer%block_size == 0 # if we do this we will have 8 layers aggregated in 1 block
# self.current_layer % (block_size//2)
# we just want to add a check
# if current_layer number exceeded the block_size, we transfer all the partial_Block to list, start a new list
# 
# each layer -> 2 layers within 

In [ ]:
# what is // operation? => floor division, so 5//2 = 2, 4//2 = 2, 3//2 = 1, 2//2 = 1, 1//2 = 0


2

In [140]:
a = torch.tensor([[1.,2.,3.], [4.,5,6]])
b = torch.tensor([4.,5,6])
c = torch.tensor([2.,1,1])
# how to make list of tensors into a single tensor? => torch.stack
torch.stack([a,b,c])
# c = torch.tensor(c)?

RuntimeError: stack expects each tensor to be equal size, but got [2, 3] at entry 0 and [3] at entry 1

In [127]:
torch.stack(c)

TypeError: expected Tensor as element 0 in argument 0, but got int